# 06 - Pipeline Definition (Bonus)

**Azure ML MLOps Workshop | Wrap-Up**

### Why Pipelines?

So far, we've done everything step-by-step in notebooks. In production MLOps,
these steps need to run automatically:
- On a schedule (weekly retraining)
- Triggered by monitoring alerts (drift detected)
- Triggered by CI/CD (new code pushed)

Azure ML Pipelines chain steps into a reproducible, schedulable DAG.

### What you'll do:
1. Define a pipeline using SDK v2
2. Understand the pipeline as a DAG
3. Submit the pipeline as a job
4. Discuss CI/CD integration

---

In [ ]:
from azure.ai.ml import MLClient, Input, Output, command, dsl
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Define Pipeline Components

Each step in the pipeline is a **component** - a self-contained unit with
defined inputs, outputs, code, and environment.

This is the code-first equivalent of Designer's drag-and-drop blocks.

In [ ]:
from azure.ai.ml.entities import Environment

custom_env = Environment(
    name="contoso-training-env",
    description="Training environment for Contoso lead classifier",
    conda_file="../../environment/conda.yml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
)

custom_env = ml_client.environments.create_or_update(custom_env)
print(f"Environment registered: {custom_env.name} v{custom_env.version}")

In [ ]:
# Define the training component
train_component = command(
    name="train_lead_classifier",
    display_name="Train Contoso Lead Classifier",
    description="Trains a text classifier on inspection comments to predict sales leads.",
    inputs={
        "input_data": Input(type="uri_file"),
        "model_name": Input(type="string", default="logistic_regression"),
        "max_features": Input(type="integer", default=5000),
        "test_size": Input(type="number", default=0.2),
    },
    outputs={
        "model_output": Output(type="uri_folder"),
    },
    code="../../src/track_a_text",
    command=(
        "python train.py "
        "--input-data ${{inputs.input_data}} "
        "--model-name ${{inputs.model_name}} "
        "--max-features ${{inputs.max_features}} "
        "--test-size ${{inputs.test_size}} "
        "--output-dir ${{outputs.model_output}}"
    ),
    environment=f"{custom_env.name}:{custom_env.version}",
)

print(f"Training component defined: {train_component.name}")

## Step 2: Define the Pipeline

The `@dsl.pipeline` decorator lets you compose components into a DAG
using plain Python. Inputs and outputs wire steps together.

In [ ]:
@dsl.pipeline(
    name="contoso-lead-classifier-pipeline",
    description="End-to-end training pipeline for Contoso inspection lead classification.",
    compute="cpu-cluster",
)
def lead_classifier_pipeline(
    training_data: Input,
    model_type: str = "logistic_regression",
    max_features: int = 5000,
):
    # Step 1: Train the model
    train_step = train_component(
        input_data=training_data,
        model_name=model_type,
        max_features=max_features,
        test_size=0.2,
    )
    train_step.display_name = f"Train {model_type}"

    return {"model_output": train_step.outputs.model_output}

print("Pipeline function defined.")

## Step 3: Submit the Pipeline Job

The pipeline runs on the compute cluster. Each step is an independent job
that can be retried, cached, and monitored.

In [ ]:
# Instantiate the pipeline with specific inputs
pipeline_job = lead_classifier_pipeline(
    training_data=Input(
        type="uri_file",
        path="azureml:classified-inspections:2",
    ),
    model_type="logistic_regression",
    max_features=5000,
)

pipeline_job.experiment_name = "contoso-lead-classifier-pipeline"

submitted_job = ml_client.jobs.create_or_update(pipeline_job)
print(f"Pipeline submitted!")
print(f"  Job name: {submitted_job.name}")
print(f"  Status: {submitted_job.status}")
print(f"  Studio URL: {submitted_job.studio_url}")

In [ ]:
# Wait for completion (optional - can also monitor in Studio)
ml_client.jobs.stream(submitted_job.name)

## Step 4: Submit Multiple Pipeline Runs

The power of pipelines: easily sweep over configurations.

In [ ]:
# Run with different model types to compare
model_types = ["random_forest", "gradient_boosting"]

for model_type in model_types:
    pipeline_job = lead_classifier_pipeline(
        training_data=Input(
            type="uri_file",
            path="azureml:classified-inspections:2",
        ),
        model_type=model_type,
        max_features=5000,
    )
    pipeline_job.experiment_name = "contoso-lead-classifier-pipeline"

    submitted = ml_client.jobs.create_or_update(pipeline_job)
    print(f"Submitted pipeline for {model_type}: {submitted.name}")

## Go to Azure ML Studio Now

Navigate to **Jobs > contoso-lead-classifier-pipeline**:

1. Click on a pipeline run to see the **DAG visualization**
2. This is the code equivalent of what Designer shows graphically
3. Click on individual steps to see their logs, metrics, and outputs
4. Compare multiple pipeline runs

---

## The Full MLOps Picture

Here's how everything we built today connects:

```
                                                   
  [Data Sources]                                   
      |                                            
      v                                            
  [Data Assets v1, v2, ...]  <-- Notebook 01       
      |                                            
      v                                            
  [Training Pipeline]       <-- This notebook (06) 
      |                                            
      v                                            
  [Experiment Tracking]     <-- Notebook 02        
      |                                            
      v                                            
  [Model Registry v1, v2]  <-- Notebook 03         
      |                                            
      v                                            
  [Online Endpoint]         <-- Notebook 04        
      |  (blue/green)                              
      v                                            
  [Model Monitoring]        <-- Notebook 05        
      |                                            
      v                                            
  [Drift Alert] --> Retrain --> New model version   
```

### CI/CD Integration (Next Steps)

To fully operationalize this with Azure DevOps or GitHub Actions:

1. **Source control:** All code (src/, config/, environment/) in Git
2. **PR trigger:** Code changes trigger a validation pipeline run
3. **Main merge:** Successful merge triggers full training pipeline
4. **Model registration:** Pipeline registers model if metrics beat baseline
5. **Deployment:** CD pipeline deploys to staging endpoint, runs integration tests
6. **Promotion:** Manual approval gates for production deployment

---

## Workshop Summary

| Topic | Azure ML Feature | Notebook |
|-------|-----------------|----------|
| Data Version Control | Data Assets | 01 |
| Experiment Tracking | MLflow + Jobs | 02 |
| Model Tracking | Model Registry | 03 |
| Model Serving | Managed Online Endpoints | 04 |
| Model Monitoring | Model Monitor | 05 |
| Pipeline Orchestration | Pipelines (SDK v2) | 06 |

All of these work together to build a production MLOps platform for Contoso
across multiple regions.